# Demo 08 — Body-based routing (notebook)

Walkthrough for **RHOAI / ODH 3.5** MaaS: gateway URL + OpenShift auth → mint API key bound to **`demo08-hybrid-catalog`** → list models → **body-based routing** (`POST /v1/chat/completions`) → optional path-based contrast → switch models on the same URL.

**Persona:** **alice** (`maas-demo-research`). Apply [Demo 08](../README.md) first (`deploy/overlays/demo08`).

**Prereq:** Python 3.9+ (stdlib only: `urllib`, `http.client`, `getpass`). IPP / payload-processing must be running for BBR.

**RHOAI:** **Demo quick swap** for `MAAS_BASE` + token between takes. Run `oc` on your laptop — not in the kernel.


## Demo quick swap

Sets demo-time overrides. Run this cell, then **§1 Configuration**.

| Variable | Effect |
|----------|--------|
| `DEMO_MAAS_BASE` | Non-empty → overrides `MAAS_BASE` env. |
| `DEMO_OPENSHIFT_TOKEN` | Non-empty → overrides `OPENSHIFT_TOKEN` env. |


In [ ]:
DEMO_MAAS_BASE = ""
DEMO_OPENSHIFT_TOKEN = ""


## 1. Configuration

Loads env (and optional OAuth password flow). Defines `MAAS_BASE`, API URLs, resolves OpenShift token.

| Variable | Required | Purpose |
|----------|----------|---------|
| `MAAS_BASE` | Yes | MaaS gateway origin (no trailing slash), e.g. `https://maas.apps…`. |
| `OPENSHIFT_TOKEN` | token *or* login | For `POST /api-keys` (alice). |
| `OPENSHIFT_USERNAME` / `OPENSHIFT_PASSWORD` | If login | OAuth basic-auth token fetch. |
| `MAAS_SUBSCRIPTION` | No | Default `demo08-hybrid-catalog`. |
| `VERIFY_TLS` | No | `1` / `true` for strict TLS. |


In [ ]:
import base64
import getpass
import http.client
import json
import os
import ssl
import urllib.error
import urllib.request
from typing import Any, Dict, Optional
from urllib.parse import parse_qs, urlencode, urlparse

_maas_demo = globals().get("DEMO_MAAS_BASE", "")
if isinstance(_maas_demo, str) and _maas_demo.strip():
    MAAS_BASE = _maas_demo.strip()
else:
    MAAS_BASE = os.environ.get("MAAS_BASE", "https://maas.YOUR_DOMAIN_HERE")

_ot_demo = globals().get("DEMO_OPENSHIFT_TOKEN", "")
if isinstance(_ot_demo, str) and _ot_demo.strip():
    OPENSHIFT_TOKEN = _ot_demo.strip()
else:
    OPENSHIFT_TOKEN = os.environ.get("OPENSHIFT_TOKEN", "").strip()

OPENSHIFT_USERNAME = os.environ.get("OPENSHIFT_USERNAME", "").strip()
OPENSHIFT_PASSWORD = os.environ.get("OPENSHIFT_PASSWORD", "").strip()
OPENSHIFT_PASSWORD_FILE = os.environ.get("OPENSHIFT_PASSWORD_FILE", "").strip()
SKIP_PASSWORD_PROMPT = os.environ.get("SKIP_PASSWORD_PROMPT", "").lower() in ("1", "true", "yes")
OPENSHIFT_OAUTH_URL = os.environ.get("OPENSHIFT_OAUTH_URL", "").strip()

SUBSCRIPTION_NAME = os.environ.get("MAAS_SUBSCRIPTION", "demo08-hybrid-catalog")
API_KEY_NAME = "demo08-notebook-key"
API_KEY_EXPIRES = "24h"
VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")


def _oauth_base_from_maas_base(maas_base: str) -> Optional[str]:
    u = urlparse(maas_base)
    host = (u.hostname or "").lower()
    if ".apps." not in host:
        return None
    suffix = host.split(".apps.", 1)[1]
    return f"{u.scheme}://oauth-openshift.apps.{suffix}"


def openshift_token_from_password(oauth_base: str, username: str, password: str, *, verify_tls: bool) -> str:
    oauth_base = oauth_base.rstrip("/")
    auth_path = "/oauth/authorize?" + urlencode(
        {"client_id": "openshift-challenging-client", "response_type": "token"}
    )
    basic = base64.b64encode(f"{username}:{password}".encode("utf-8")).decode("ascii")
    u = urlparse(oauth_base)
    if u.scheme != "https" or not u.hostname:
        raise ValueError("OPENSHIFT_OAUTH_URL must be https://… with a hostname")
    ctx = ssl.create_default_context()
    if not verify_tls:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    conn = http.client.HTTPSConnection(u.hostname, port=u.port or 443, context=ctx, timeout=120)
    conn.request(
        "GET",
        auth_path,
        headers={
            "Authorization": f"Basic {basic}",
            "X-CSRF-Token": "notebook",
            "Accept": "*/*",
            "User-Agent": "maas-demo08-notebook/1.0",
        },
    )
    resp = conn.getresponse()
    resp.read()
    location = resp.getheader("Location")
    conn.close()
    if resp.status not in (302, 303, 307) or not location:
        raise RuntimeError(
            f"OAuth login failed (HTTP {resp.status}). Prefer OPENSHIFT_TOKEN (oc whoami -t)."
        )
    frag = urlparse(location).fragment
    token = parse_qs(frag).get("access_token", [None])[0] if frag else None
    if not token:
        raise RuntimeError(f"No access_token in OAuth redirect: {location!r}")
    return token


MAAS_BASE = MAAS_BASE.rstrip("/")
API_KEYS_URL = f"{MAAS_BASE}/maas-api/v1/api-keys"
MODELS_URL = f"{MAAS_BASE}/maas-api/v1/models"
BBR_CHAT_URL = f"{MAAS_BASE}/v1/chat/completions"
_oauth_base = OPENSHIFT_OAUTH_URL or _oauth_base_from_maas_base(MAAS_BASE)

if OPENSHIFT_TOKEN:
    print("Using OPENSHIFT_TOKEN from the environment (username/password skipped).")
elif OPENSHIFT_USERNAME:
    _pw = OPENSHIFT_PASSWORD
    if not _pw and OPENSHIFT_PASSWORD_FILE:
        with open(OPENSHIFT_PASSWORD_FILE, encoding="utf-8") as _f:
            _pw = _f.readline().strip()
    if not _pw and not SKIP_PASSWORD_PROMPT:
        _pw = getpass.getpass("OpenShift password (hidden; not echoed): ")
    if not _pw:
        raise SystemExit("Set OPENSHIFT_PASSWORD, OPENSHIFT_PASSWORD_FILE, or OPENSHIFT_TOKEN.")
    if not _oauth_base:
        raise SystemExit("Could not derive OAuth URL from MAAS_BASE. Set OPENSHIFT_OAUTH_URL.")
    OPENSHIFT_TOKEN = openshift_token_from_password(
        _oauth_base, OPENSHIFT_USERNAME, _pw, verify_tls=VERIFY_TLS
    )
    print("Fetched OPENSHIFT_TOKEN via username/password (OAuth).")
else:
    print("No OPENSHIFT_TOKEN or OPENSHIFT_USERNAME — set one before creating an API key.")

print("MAAS_BASE      :", MAAS_BASE)
print("BBR_CHAT_URL   :", BBR_CHAT_URL)
print("SUBSCRIPTION   :", SUBSCRIPTION_NAME)
print("VERIFY_TLS     :", VERIFY_TLS)
print("Token ready    :", bool(OPENSHIFT_TOKEN))


## 2. Create an API key

`POST /maas-api/v1/api-keys` with OpenShift bearer token, bound to **`demo08-hybrid-catalog`**. Response `key` → **`API_KEY`** (shown once).


In [ ]:
def http_json(
    method: str,
    url: str,
    *,
    token: Optional[str] = None,
    data: Optional[Dict[str, Any]] = None,
):
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    body = json.dumps(data).encode("utf-8") if data is not None else None
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            raw = resp.read().decode("utf-8")
            return resp.status, json.loads(raw) if raw else {}
    except urllib.error.HTTPError as e:
        err_body = e.read().decode("utf-8", errors="replace")
        try:
            parsed = json.loads(err_body) if err_body else {}
        except json.JSONDecodeError:
            parsed = {"_raw": err_body}
        raise RuntimeError(f"HTTP {e.code}: {parsed}") from None


if not OPENSHIFT_TOKEN:
    raise SystemExit("Set OPENSHIFT_TOKEN (alice) to create an API key.")

payload = {
    "name": API_KEY_NAME,
    "description": "Demo 08 body-based routing notebook",
    "expiresIn": API_KEY_EXPIRES,
    "subscription": SUBSCRIPTION_NAME,
}
_, body = http_json("POST", API_KEYS_URL, token=OPENSHIFT_TOKEN, data=payload)
API_KEY = body.get("key")
if not API_KEY:
    raise RuntimeError(f"No 'key' in response: {body}")
print("API key created (prefix):", API_KEY[:20] + "…")


## 3. Model discovery

`GET /maas-api/v1/models` with **`API_KEY`**. Pick Granite (on-cluster) and optionally `sim-chat` (external llm-katan) for later cells.


In [ ]:
_B, _R = "\033[1m", "\033[0m"

_, models_body = http_json("GET", MODELS_URL, token=API_KEY)
data = models_body.get("data") or []
if not data:
    raise SystemExit("No models; apply Demo 08 and wait for MaaSModelRef Ready.")

def _pick(pred):
    for m in data:
        mid = (m.get("id") or m.get("name") or "")
        if pred(mid.lower()):
            return m
    return None

granite = _pick(lambda s: "granite" in s)
sim_chat = _pick(lambda s: "sim-chat" in s and "sim-chat-2" not in s)
first = granite or data[0]

MODELS_BY_ID = {(m.get("id") or m.get("name")): m for m in data if m.get("id") or m.get("name")}
GRANITE_ID = (granite or first).get("id") or (granite or first).get("name")
GRANITE_URL = ((granite or first).get("url") or "").rstrip("/")
SIM_CHAT_ID = (sim_chat.get("id") or sim_chat.get("name")) if sim_chat else None
SIM_CHAT_URL = (sim_chat.get("url") or "").rstrip("/") if sim_chat else None

print()
print("┌" + "─" * 62 + "┐")
print("│  " + _B + "GRANITE_ID " + _R + " " + str(GRANITE_ID)[:46].ljust(46) + " │")
print("│  " + _B + "SIM_CHAT   " + _R + " " + str(SIM_CHAT_ID or "(not in catalog)").ljust(46)[:46] + " │")
print("│  " + _B + "BBR URL    " + _R + " " + BBR_CHAT_URL[:46].ljust(46) + " │")
print("└" + "─" * 62 + "┘")
print()
print(_B + "All models (id → url)" + _R)
for m in data:
    print(" -", m.get("id") or m.get("name"), "→", m.get("url"))


## 4. Body-based routing (preferred)

Single endpoint: **`POST {MAAS_BASE}/v1/chat/completions`**. Model selection is **only** the JSON `model` field — OpenAI SDK–friendly (`base_url` = `{MAAS_BASE}/v1`).


In [ ]:
USER_MESSAGE = "Say hello in one short sentence."
MAX_TOKENS = 64


def chat_completion(*, url: str, model: str, user_message: str, max_tokens: int = 64):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": user_message}],
        "max_tokens": max_tokens,
    }
    status, body = http_json("POST", url, token=API_KEY, data=payload)
    choices = body.get("choices") or []
    msg = (choices[0].get("message") or {}) if choices and isinstance(choices[0], dict) else {}
    content = msg.get("content") or ""
    usage = body.get("usage") or {}
    print("HTTP", status, "| model:", model)
    print("Assistant:", content if content else "(empty)")
    if usage:
        print("usage:", usage)
    print()
    return status, body


print("=== BBR → Granite (on-cluster simulator) ===")
chat_completion(url=BBR_CHAT_URL, model=GRANITE_ID, user_message=USER_MESSAGE, max_tokens=MAX_TOKENS)

if SIM_CHAT_ID:
    print("=== BBR → sim-chat (llm-katan external) ===")
    try:
        chat_completion(url=BBR_CHAT_URL, model=SIM_CHAT_ID, user_message=USER_MESSAGE, max_tokens=MAX_TOKENS)
    except RuntimeError as e:
        print("External call failed (check IPP + reachability of 3-132-132-211.sslip.io):", e)
else:
    print("(Skip external BBR — sim-chat not listed; check ExternalModel / MaaSModelRef Ready.)")


## 5. Path-based contrast (still supported)

Same chat body, but URL is **`{MODEL_URL}/v1/chat/completions`** from discovery. Useful if IPP is unavailable; prefer BBR for agents and SDKs.


In [ ]:
if not GRANITE_URL:
    raise SystemExit("No path-based MODEL_URL for Granite.")

PATH_CHAT_URL = f"{GRANITE_URL}/v1/chat/completions"
print("=== Path-based → Granite ===")
print("POST", PATH_CHAT_URL)
chat_completion(url=PATH_CHAT_URL, model=GRANITE_ID, user_message=USER_MESSAGE, max_tokens=MAX_TOKENS)


## 6. Same `base_url`, switch models

Demonstrates why BBR matters for OpenAI-compatible clients: one URL, change only `model`.


In [ ]:
print("OpenAI-compatible base_url:", f"{MAAS_BASE}/v1")
print("Chat path:               ", "/chat/completions")
print()

for label, mid in (("on-cluster", GRANITE_ID), ("external", SIM_CHAT_ID)):
    if not mid:
        print(f"skip {label}: no model id")
        continue
    print(f"--- switch model → {mid} ({label}) ---")
    try:
        chat_completion(
            url=BBR_CHAT_URL,
            model=mid,
            user_message=f"Reply with only the word '{label}'.",
            max_tokens=16,
        )
    except RuntimeError as e:
        print(f"{label} failed:", e)
